In [1]:
!pip install -q transformers dill scikit-learn

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. Enable Runtime > Change runtime type > T4 GPU.")

CUDA available: True
GPU: Tesla T4


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path

DRIVE_DATASET = Path(
    "/content/drive/MyDrive/"
    "phishing_models/vonDataset20180426.dill"
)

DRIVE_OUTPUT = Path(
    "/content/drive/MyDrive/"
    "phishing_models/urlbert_vonpower_full"
)

DRIVE_OUTPUT.mkdir(
    parents=True,
    exist_ok=True,
)

print(DRIVE_DATASET)
print("Exists:", DRIVE_DATASET.exists())

/content/drive/MyDrive/phishing_models/vonDataset20180426.dill
Exists: True


In [5]:

import random
import time
import dill
import numpy as np
import torch

from torch.utils.data import (
    DataLoader,
    IterableDataset,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)


MODEL_NAME = (
    "CrabInHoney/"
    "urlbert-tiny-v4-phishing-classifier"
)

MAX_LENGTH = 64
CHUNK_OVERLAP = 16
BATCH_SIZE = 32
EPOCHS = 1
LEARNING_RATE = 2e-5
NUM_WORKERS = 2
SEED = 42


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

print("Loading .dill dataset...")

with DRIVE_DATASET.open("rb") as file:
    data = dill.load(file)

train_x = np.asarray(data["train_x"])
train_y = np.asarray(data["train_y"]).reshape(-1)
char_to_int = data["char_to_int"]

reverse_char_to_int = {
    int(value): str(key)
    for key, value in char_to_int.items()
}

print("Train records:", len(train_x))
print("Train labels:", np.bincount(train_y))
print("Train shape:", train_x.shape)

print("Loading URLBERT tokenizer and model...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = (
    AutoModelForSequenceClassification
    .from_pretrained(MODEL_NAME)
)

model.to(device)
model.train()

print("Model loaded.")

Device: cuda
Loading .dill dataset...
Train records: 1247489
Train labels: [640064 607425]
Train shape: (1247489, 150)
Loading URLBERT tokenizer and model...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/10.0k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.8MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

Model loaded.


In [6]:
def decode_url(row):
    characters = []

    for token in row:
        token = int(token)

        if token == 0:
            continue

        characters.append(
            reverse_char_to_int.get(token, "")
        )

    return "".join(characters)


def create_chunks(url):
    tokens = tokenizer.tokenize(url)

    token_ids = tokenizer.convert_tokens_to_ids(
        tokens
    )

    content_length = MAX_LENGTH - 2

    if len(token_ids) <= content_length:
        chunks = [token_ids]
    else:
        step = content_length - CHUNK_OVERLAP
        chunks = []

        for start in range(
            0,
            len(token_ids),
            step,
        ):
            chunk = token_ids[
                start:start + content_length
            ]

            if chunk:
                chunks.append(chunk)

            if start + content_length >= len(
                token_ids
            ):
                break

    result = []

    for chunk in chunks:
        input_ids = [
            tokenizer.cls_token_id,
            *chunk,
            tokenizer.sep_token_id,
        ]

        attention_mask = [1] * len(input_ids)

        padding_length = (
            MAX_LENGTH - len(input_ids)
        )

        input_ids.extend(
            [tokenizer.pad_token_id or 0]
            * padding_length
        )

        attention_mask.extend(
            [0] * padding_length
        )

        result.append(
            (
                input_ids,
                attention_mask,
            )
        )

    return result


class VonpowerChunkDataset(IterableDataset):
    def __init__(self, indices):
        self.indices = indices

    def __iter__(self):
        worker_info = (
            torch.utils.data.get_worker_info()
        )

        indices = self.indices

        if worker_info is not None:
            indices = indices[
                worker_info.id::worker_info.num_workers
            ]

        for index in indices:
            url = decode_url(train_x[index])
            label = int(train_y[index])

            chunks = create_chunks(url)

            for input_ids, attention_mask in chunks:
                yield {
                    "input_ids": torch.tensor(
                        input_ids,
                        dtype=torch.long,
                    ),
                    "attention_mask": torch.tensor(
                        attention_mask,
                        dtype=torch.long,
                    ),
                    "labels": torch.tensor(
                        label,
                        dtype=torch.long,
                    ),
                }

In [7]:
from transformers import default_data_collator

train_indices = np.arange(
    len(train_x),
    dtype=np.int64,
)

rng = np.random.default_rng(SEED)
rng.shuffle(train_indices)

dataset = VonpowerChunkDataset(
    train_indices
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    collate_fn=default_data_collator,
    pin_memory=torch.cuda.is_available(),
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)

start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    batch_count = 0

    for batch in loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        batch_count += 1

        if batch_count % 500 == 0:
            print(
                f"Epoch {epoch + 1}, "
                f"Batch {batch_count}, "
                f"Loss: "
                f"{total_loss / batch_count:.4f}"
            )

    print(
        f"Epoch {epoch + 1} finished. "
        f"Average loss: "
        f"{total_loss / batch_count:.4f}"
    )

training_seconds = time.time() - start_time

print(
    f"Training time: "
    f"{training_seconds / 60:.2f} minutes"
)

/tmp/ipykernel_1121/3437264987.py:28: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (70 > 64). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (77 > 64). Running this sequence through the model will result in indexing errors
/tmp/ipykernel_1121/3437264987.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 1, Batch 500, Loss: 0.2046
Epoch 1, Batch 1000, Loss: 0.1763
Epoch 1, Batch 1500, Loss: 0.1614
Epoch 1, Batch 2000, Loss: 0.1544
Epoch 1, Batch 2500, Loss: 0.1475
Epoch 1, Batch 3000, Loss: 0.1430
Epoch 1, Batch 3500, Loss: 0.1389
Epoch 1, Batch 4000, Loss: 0.1361
Epoch 1, Batch 4500, Loss: 0.1327
Epoch 1, Batch 5000, Loss: 0.1296
Epoch 1, Batch 5500, Loss: 0.1269
Epoch 1, Batch 6000, Loss: 0.1243
Epoch 1, Batch 6500, Loss: 0.1220
Epoch 1, Batch 7000, Loss: 0.1199
Epoch 1, Batch 7500, Loss: 0.1185
Epoch 1, Batch 8000, Loss: 0.1165
Epoch 1, Batch 8500, Loss: 0.1149
Epoch 1, Batch 9000, Loss: 0.1135
Epoch 1, Batch 9500, Loss: 0.1122
Epoch 1, Batch 10000, Loss: 0.1109
Epoch 1, Batch 10500, Loss: 0.1098
Epoch 1, Batch 11000, Loss: 0.1086
Epoch 1, Batch 11500, Loss: 0.1074
Epoch 1, Batch 12000, Loss: 0.1061
Epoch 1, Batch 12500, Loss: 0.1051
Epoch 1, Batch 13000, Loss: 0.1040
Epoch 1, Batch 13500, Loss: 0.1032
Epoch 1, Batch 14000, Loss: 0.1023
Epoch 1, Batch 14500, Loss: 0.1016
Epoch

In [8]:
model.eval()

model.save_pretrained(
    DRIVE_OUTPUT
)

tokenizer.save_pretrained(
    DRIVE_OUTPUT
)

print(
    "Final model saved to:"
)
print(DRIVE_OUTPUT)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to:
/content/drive/MyDrive/phishing_models/urlbert_vonpower_full
